In [1]:
import pandas as pd
import numpy as np

from aicspylibczi import CziFile

In [2]:
czi = CziFile("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/organoids_h&e/czi/amadurga-10-10-2025-003.czi")
region_df = pd.read_csv("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/data/xenium/metadata/Regions_coordinates_18samples.csv")

In [5]:
czi_bbox = czi.get_scene_bounding_box()
region_df = pd.read_csv("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/data/xenium/metadata/Regions_coordinates_18samples.csv")

def add_hne_bboxes_to_region_df(czi_bbox, region_df):
    # note: xs and ys are flipped between region_df and czi_df 
    # so czi_xs will be a function of xenium ys and visa versa
    xenium_expansion_x = 1000
    xenium_expansion_y = 1000

    old_xenium_ymin = region_df['y_min'].min()
    old_xenium_ymax = region_df['y_max'].max()
    old_xenium_xmin = region_df['x_min'].min()
    old_xenium_xmax = region_df['x_max'].max()

    # region_df['x_min'] += xenium_shift_x
    region_df['x_max'] += xenium_expansion_x
    region_df['x_min'] -= xenium_expansion_x
    region_df['y_max'] += xenium_expansion_y
    region_df['y_min'] -= xenium_expansion_y

    region_df['x_min'] = np.clip(region_df['x_min'], old_xenium_xmin, old_xenium_xmax)
    region_df['x_max'] = np.clip(region_df['x_max'], old_xenium_xmin, old_xenium_xmax)
    region_df['y_min'] = np.clip(region_df['y_min'], old_xenium_ymin, old_xenium_ymax)
    region_df['y_max'] = np.clip(region_df['y_max'], old_xenium_ymin, old_xenium_ymax)

    xenium_xmin = region_df['x_min'].min()
    xenium_xmax = region_df['x_max'].max()
    xenium_ymin = region_df['y_min'].min()
    xenium_ymax = region_df['y_max'].max()

    xenium_width = xenium_xmax - xenium_xmin
    xenium_height = xenium_ymax - xenium_ymin
    xenium_x_to_czi_y_scale_factor = czi_bbox.h / xenium_width
    xenium_y_to_czi_x_scale_factor = czi_bbox.w / xenium_height
    xenium_x_to_czi_y_shift_factor = czi_bbox.y
    xenium_y_to_czi_x_shift_factor = czi_bbox.x

    xenium_x_to_czi_y = lambda xenium_x: xenium_x * xenium_x_to_czi_y_scale_factor + xenium_x_to_czi_y_shift_factor
    xenium_y_to_czi_x = lambda xenium_y: (old_xenium_ymax - xenium_y) * xenium_y_to_czi_x_scale_factor + xenium_y_to_czi_x_shift_factor

    region_df['czi_xmin'] = region_df['y_max'].apply(xenium_y_to_czi_x)
    region_df['czi_xmax'] = region_df['y_min'].apply(xenium_y_to_czi_x)
    region_df['czi_width'] = region_df['czi_xmax'] - region_df['czi_xmin']

    region_df['czi_ymin'] = region_df['x_min'].apply(xenium_x_to_czi_y)
    region_df['czi_ymax'] = region_df['x_max'].apply(xenium_x_to_czi_y)
    region_df['czi_height'] = region_df['czi_ymax'] - region_df['czi_ymin']

    # print(region_df[region_df['Name'] == "1CFV"].head())

    return region_df.copy()

updated_region_df = add_hne_bboxes_to_region_df(czi_bbox, region_df)
' '.join(updated_region_df[updated_region_df['Need_40_cohort']=='Yes']['Name'])

'1HVQ_big 1HVQ_big_CAFs 0WFQ_big 1DDI 07WM 1CFV 1HVC OUC4 12I1'

In [ ]:
from tqdm import tqdm

import tifffile

pth = "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ome_tiff/run_4_1"


for index, row in tqdm(updated_region_df.iterrows()):
    # if row['Name'] != "1CFV":
    #     continue

    im = czi.read_mosaic(region=(
        int(row['czi_xmin']), 
        int(row['czi_ymin']),
        int(row['czi_width']), 
        int(row['czi_height']), 
    ), C=0)[0]

    tifffile.imwrite(f"{pth}/{row['Name']}.ome.tiff", im)

    print(row['Name'])